# SAP AI: Colab training

Select **Runtime → Change runtime type → T4 GPU**. This notebook keeps datasets and checkpoints in Google Drive so a disconnected runtime can resume.

In [ ]:
REPO_URL = "https://github.com/lgtyqz/sapai-python.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
DRIVE_RUN_DIR = "/content/drive/MyDrive/sapai-runs/run-001"  # @param {type:"string"}
BOARDS_JSONL = "/content/drive/MyDrive/sapai-data/boards.jsonl"  # @param {type:"string"}
SEED = 2026  # @param {type:"integer"}
RUN_FULL_TRAINING = False  # @param {type:"boolean"}
assert REPO_URL, "Set REPO_URL to the Git repository containing this project."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path('/content/sapai-python')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print(repo)

In [ ]:
%pip install -q -e '.[ml,dev]'

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert Path(BOARDS_JSONL).exists(), f'Missing stable board dataset: {BOARDS_JSONL}'
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
subprocess.run(['python', '-m', 'sapai.cli', 'model-smoke'], check=True)

## Small end-to-end smoke run

This validates labeling, both checkpoint loops, complete Arena rollouts, and search distillation before a long run. It still needs enough replay IDs to form all three splits.

In [ ]:
smoke_dir = str(Path(DRIVE_RUN_DIR).with_name(Path(DRIVE_RUN_DIR).name + '-smoke'))
smoke = [
    'python', '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_JSONL, '--workdir', smoke_dir,
    '--battle-examples', '100', '--simulations-per-pair', '1',
    '--battle-epochs', '1', '--bootstrap-episodes', '2',
    '--bootstrap-epochs', '1', '--search-episodes', '1',
    '--search-epochs', '1', '--search-simulations', '4',
    '--search-candidates', '4', '--batch-size', '32', '--seed', str(SEED),
]
subprocess.run(smoke, check=True)

## Full training sequence

Edit counts here for the dataset and time budget. Rerunning resumes from Drive checkpoints.

In [ ]:
full = [
    'python', '-m', 'sapai.cli', 'train-sequence',
    '--boards', BOARDS_JSONL, '--workdir', DRIVE_RUN_DIR,
    '--battle-examples', '100000', '--simulations-per-pair', '8',
    '--battle-epochs', '20', '--bootstrap-episodes', '1000',
    '--bootstrap-epochs', '20', '--search-episodes', '250',
    '--search-epochs', '5', '--search-simulations', '32',
    '--search-candidates', '8', '--batch-size', '128', '--seed', str(SEED),
]
if RUN_FULL_TRAINING:
    subprocess.run(full, check=True)
else:
    print('Set RUN_FULL_TRAINING=True when the smoke run succeeds.')

## Visualize the latest policy

The generated HTML contains both shops and battles and is also saved with the run artifacts.

In [ ]:
from IPython.display import HTML, display
active_run = DRIVE_RUN_DIR if RUN_FULL_TRAINING else smoke_dir
visualization = str(Path(active_run) / 'arena.html')
subprocess.run([
    'python', '-m', 'sapai.cli', 'visualize-arena',
    '--boards', BOARDS_JSONL, '--policy', 'model',
    '--policy-weights', str(Path(active_run) / 'policy-model'),
    '--seed', str(SEED), '--output', visualization,
], check=True)
display(HTML(filename=visualization))